# Multiagent: 교육과정 팀의 작업을 실시간으로 관찰하기

subagent에 위임하는 coordinator에는 관찰의 공백이 있습니다. 세션 수준 stream은 model이 생성하는 primary thread text를 미리 보여 주지만, subagent의 출력은 전체 turn이 buffer에 쌓인 뒤에야 보입니다. researcher가 2분 동안 실행된다면 2분 동안 아무것도 볼 수 없습니다.

이 notebook은 Managed Agents API 기능 네 가지를 함께 사용하여 이 공백을 메웁니다.

- **thread별 delta streaming.** 각 세션 thread의 stream은 세션 수준 stream과 같은 `event_deltas` parameter를 받으므로 subagent의 text가 자체 stream에서 실시간으로 미리 표시됩니다.
- **세션 생성 시 initial event.** `sessions.create`가 `initial_events`를 받으므로 생성하는 호출에서 바로 세션이 작업을 시작합니다.
- **agent model의 effort.** `model.effort`는 agent별로 각 inference 호출에서 Claude가 얼마나 깊이 작업할지 설정합니다. 팀에서는 역할별 비용 조절 수단입니다.
- **agent update의 선택적 version.** `agents.update`는 현재 version을 필수가 아닌 선택적인 concurrency key로 취급합니다.

만들 팀은 7학년 과학의 일주일 단원을 계획합니다. coordinator가 교육 기준 researcher(web search, high effort)와 수업 writer(web 접근 없음)에 위임한 뒤 단원 계획을 구성합니다. `multiagent` coordinator 패턴이 처음이라면 [`CMA_coordinate_specialist_team.ipynb`](CMA_coordinate_specialist_team.ipynb)부터 시작하세요. 이 notebook은 해당 형태를 기반으로 합니다.


## 1. client 설정하기

이 기능들은 표준 `managed-agents-2026-04-01` beta header를 사용합니다. 이 notebook의 SDK 호출(`sessions.create`의 `initial_events`, thread stream의 `event_deltas`, `model.effort`)에는 `anthropic>=0.118.0`이 필요합니다.


In [ ]:
%%capture
%pip install -qU "anthropic>=0.118.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. specialist 두 명 생성하기

researcher는 web search와 `high` effort 수준을 받습니다. 교육 기준에 맞추는 일은 판단이 많이 필요하기 때문입니다. 찾은 모든 것을 나열하는 대신 어떤 수행 기대가 단원에 실제로 맞는지 따져야 합니다. `effort`는 `model` 객체 안에 들어가고 `low`, `medium`, `high`, `xhigh`, `max`를 받습니다(단순 문자열 또는 `{"type": "high"}`). 모든 model이 모든 수준을 지원하지 않으며 유효하지 않은 조합은 생성 시 거부됩니다.

Effort는 팀의 주요 비용 조절 수단이기도 합니다. 높은 수준에서는 Claude가 inference 호출당 더 많은 token을 쓸 수 있고 낮은 수준은 이를 제한합니다. agent별로 설정되므로 필요한 역할에만 깊이를 위한 비용을 지불합니다. 더 큰 팀에서는 formatting이나 triage 역할을 `low`로 낮추면서 researcher의 budget은 건드리지 않을 수 있습니다.

effort는 세션별이 아니라 agent에 설정하세요. 세션별 `model` override 안의 `effort` 수준은 적용되지 않습니다.

web tool 설정에는 허용 및 차단 domain 목록도 지정할 수 있으므로 production deployment에서는 이 researcher가 신뢰해야 할 교육 기준 사이트로 범위를 제한할 수 있습니다. 설정 형태는 [toolset 설정](https://platform.claude.com/docs/en/managed-agents/tools#configuring-the-toolset)을 참고하세요.


In [ ]:
RESEARCHER_SYSTEM = """You research US middle school science standards.
Given a unit topic and grade level, use web search to find:
- The two or three NGSS performance expectations the unit should target
- What students are expected to already know, and where the topic leads next
- Two or three documented student misconceptions for the topic
Return via send_to_parent:
{"standards": [...], "prior_knowledge": ..., "leads_to": ...,
 "misconceptions": [...], "sources": [...]}"""

standards_researcher = client.beta.agents.create(
    name="standards_researcher",
    description="Finds the NGSS standards and documented misconceptions for a science topic.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL, "effort": "high"},
    system=RESEARCHER_SYSTEM,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
    betas=BETAS,
)

수업 writer는 coordinator가 전달하는 내용만으로 작업합니다. web tool을 비활성화하면 새 검색 결과가 아니라 researcher가 검증한 조사 결과를 근거로 수업을 만들게 됩니다.


In [ ]:
WRITER_SYSTEM = """You write middle school lesson plans.
You will be given a topic, grade level, target standards, and known misconceptions.
Draft five 45-minute lessons, one per day. For each day give:
- Objective (tied to a target standard)
- Warm-up (5 min)
- Main activity (30 min)
- Exit ticket (one question that probes a misconception where possible)
Return via send_to_parent: {"days": [{"day": 1, "objective": ..., "warm_up": ...,
"main_activity": ..., "exit_ticket": ...}, ...]}"""

lesson_writer = client.beta.agents.create(
    name="lesson_writer",
    description="Drafts a day-by-day lesson sequence from standards and misconceptions.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL},
    system=WRITER_SYSTEM,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
    betas=BETAS,
)

생성 응답은 생략한 필드를 포함하여 결정된 model 설정을 다시 보여 줍니다. researcher에는 설정한 `high`가 나타나고 writer에는 model의 기본 effort가 나타납니다. `effort`가 `None`으로 돌아오면 조직의 beta header가 아직 이 기능을 제공하지 않는 것입니다. 필드가 거부되는 대신 제외되므로 이 응답에서 이를 확인할 수 있습니다.


In [ ]:
for agent in (standards_researcher, lesson_writer):
    effort = getattr(agent.model, "effort", None)
    print(f"{agent.name}: {agent.id} v{agent.version} effort={effort and effort.type}")

## 3. coordinator 생성하기

coordinator는 `multiagent` roster를 가지고 있습니다. spawn 및 delegation tool은 roster에서 자동으로 주입되며 roster는 각 child를 현재 version으로 고정합니다. 나중에 researcher를 update할 때 이 고정이 중요합니다.


In [ ]:
COORDINATOR_SYSTEM = """You plan one-week teaching units.
Given a topic and grade level:
1. Send the topic and grade to standards_researcher.
2. When the researcher reports back, send the topic, grade, target standards, and
misconceptions to lesson_writer.
3. Write /mnt/session/outputs/unit_plan.md with sections: Overview, Standards alignment,
Day-by-day plan (from the writer), Misconceptions to watch for, Materials.
Keep it under three pages. To revise the plan, rewrite the whole file with the write
tool rather than patching it with edit. While a subagent is working, wait silently: no
status commentary until its report arrives."""

coordinator = client.beta.agents.create(
    name="unit_planner",
    description="Plans one-week teaching units by delegating research and drafting.",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL},
    system=COORDINATOR_SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [standards_researcher.id, lesson_writer.id],
    },
    betas=BETAS,
)
print(f"{coordinator.name}: {coordinator.id} v{coordinator.version}")

## 4. 첫 message를 함께 넣어 세션 시작하기

`initial_events` 없이 세션을 시작하려면 두 번 호출합니다. 생성하고 `user.message`를 게시합니다. 생성 시 첫 message를 넣으면 하나로 합쳐집니다. 배열은 최대 50개의 `user.message`와 `user.define_outcome` event를 받아 순서대로 처리하며, 검증은 전부 성공하거나 전부 실패하는 방식입니다. event 하나라도 실패하면 세션이 생성되지 않습니다. 비어 있지 않은 목록은 같은 호출에서 agent 루프를 시작하므로 세션은 이미 `running`을 향해 움직이는 상태로 반환됩니다.

단원 계획을 rubric으로 평가하려면 여기에 `user.define_outcome` event 하나를 추가하세요(`rubric`이 필수이며 생성당 하나만 허용됩니다). [`CMA_verify_with_outcome_grader.ipynb`](CMA_verify_with_outcome_grader.ipynb)에서 해당 루프를 다룹니다.


In [ ]:
import time

env = client.beta.environments.create(
    name="curriculum-studio",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=env.id,
    title="Unit plan: water cycle, grade 7",
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Plan a one-week unit on the water cycle for 7th grade science. "
                    "Classes are 45 minutes. Write the plan to /mnt/session/outputs/unit_plan.md.",
                }
            ],
        }
    ],
    betas=BETAS,
)

deadline = time.monotonic() + 60
while session.status == "idle" and time.monotonic() < deadline:
    time.sleep(1)
    session = client.beta.sessions.retrieve(session.id, betas=BETAS)

print(f"{session.id}: {session.status}")
if session.status == "idle":
    events = client.beta.sessions.events.list(session.id, limit=1, betas=BETAS)
    if next(iter(events), None) is None:
        raise RuntimeError(
            "The session has no events: your organization's beta header does not carry "
            "initial_events yet. The field is dropped at the gateway, not rejected."
        )
    raise RuntimeError(
        "The seeded message is in the event log but the session is still idle after 60s. "
        "Re-run this cell to keep waiting."
    )

## 5. 전체 팀 실시간으로 관찰하기

preview는 의도적으로 thread 범위입니다. 연결은 읽는 thread만 미리 보여 줍니다. 세션 수준 stream은 primary thread를 미리 보여 주고, child의 preview는 `GET /v1/sessions/{session_id}/threads/{thread_id}/stream`의 자체 stream에만 나타납니다. 따라서 패턴은 thread당 stream 하나입니다. coordinator를 위해 세션 수준 stream을 읽고, `session.thread_created` event가 child를 알릴 때마다(`session_thread_id`와 `agent_name` 포함) background thread에서 해당 thread stream에 watcher를 연결합니다.

코드를 결정하는 규칙은 두 가지입니다.

- **preview는 임시 buffer이며 buffer에 쌓인 event가 기록입니다.** delta는 best-effort이며 부하가 높으면 멈출 수 있습니다. 뒤따르는 buffer에 쌓인 `agent.message`에 전체 내용이 담깁니다. SDK의 `accumulate_managed_agents_event` helper는 `event_start`, `event_delta`, buffer에 쌓인 event를 하나의 snapshot으로 합치며 기록이 도착하면 preview를 교체합니다. stream 연결마다 accumulator 하나를 실행하세요.
- **replay는 없습니다.** model 요청이 시작된 뒤 열린 연결은 진행 중인 요청의 delta를 받지 못하며 재연결해도 놓친 delta는 replay되지 않습니다. `session.thread_created`가 도착하는 즉시 watcher를 연결하면 child의 첫 응답부터 작업을 볼 수 있습니다.

각 watcher는 child의 turn이 끝날 때 발생하는 `session.thread_status_idle`에서 종료합니다.


In [ ]:
import threading

from anthropic.lib.sessions import accumulate_managed_agents_event

print_lock = threading.Lock()
current_speaker = None
reconciliations = []


def say(speaker, text):
    """streaming text를 출력하며 화자가 바뀔 때마다 이름을 표시합니다."""
    global current_speaker
    with print_lock:
        if speaker != current_speaker:
            print(f"\n\n=== {speaker} ===")
            current_speaker = speaker
        print(text, end="", flush=True)


def text_of(event):
    if event is None:
        return ""
    return "".join(block.text for block in event.content if block.type == "text")


def fold_preview(snapshot, ev, speaker):
    """preview event 하나를 화자의 snapshot에 합치고 새 snapshot을 반환합니다.

    delta text를 도착하는 대로 출력합니다. buffer의 기록이 도착하면 누적된 preview와
    비교하는 reconciliation 행을 기록합니다. id 확인은 message의 모든 delta가
    버려지는 경우에 대비합니다. 그때 snapshot에는 이번 message의 preview 대신
    이전 message가 여전히 들어 있습니다.
    """
    if ev.type == "event_delta":
        say(speaker, ev.delta.content.text)
    elif ev.type == "agent.message":
        preview = text_of(snapshot) if snapshot and snapshot.id == ev.id else ""
        final = text_of(ev)
        reconciliations.append((speaker, len(preview), len(final), final.startswith(preview)))
    return accumulate_managed_agents_event(snapshot, ev)


def watch_child(thread_id, agent_name):
    """child thread 하나를 streaming하며 model이 생성하는 text를 미리 보여 줍니다."""
    snapshot = None
    with client.beta.sessions.threads.events.stream(
        thread_id,
        session_id=session.id,
        event_deltas=["agent.message"],
        betas=BETAS,
    ) as thread_stream:
        for ev in thread_stream:
            if ev.type in ("event_start", "event_delta", "agent.message"):
                snapshot = fold_preview(snapshot, ev, agent_name)
            elif ev.type == "agent.tool_use":
                detail = ev.input.get("query", "") if ev.name == "web_search" else ""
                say(agent_name, f"\n[{ev.name}] {detail}")
            elif ev.type in ("session.thread_status_idle", "session.thread_status_terminated"):
                break

main 루프는 coordinator text에 같은 `fold_preview`를 적용하고, primary thread에만 나타나는 coordination event도 처리합니다. child가 생성될 때의 `session.thread_created`, coordinator가 작업을 넘길 때의 `agent.thread_message_sent`, child가 보고할 때의 `agent.thread_message_received`입니다. child thread에서 교차 게시된 tool 호출에는 `session_thread_id`가 들어 있으므로 루프는 이를 건너뜁니다. child의 자체 watcher가 보여 줍니다.

루프는 모든 `session.status_idle`에서 끝나며 사유가 `end_turn`이 아니면 종료 사유를 출력합니다. tool 확인을 기다리거나 재시도 횟수를 소진한 세션은 셀을 멈추게 해야지 무한 대기하게 해서는 안 됩니다. 두 루프는 terminated 상태 event에서도 종료하므로 복구할 수 없는 세션 오류가 stream을 열어 둔 채 남기는 대신 셀을 끝냅니다.


In [ ]:
watchers = []
snapshot = None

with client.beta.sessions.events.stream(
    session.id, event_deltas=["agent.message"], betas=BETAS
) as stream:
    for ev in stream:
        if ev.type in ("event_start", "event_delta", "agent.message"):
            snapshot = fold_preview(snapshot, ev, "unit_planner")
        elif ev.type == "session.thread_created":
            say("unit_planner", f"\n[spawned {ev.agent_name}]")
            watcher = threading.Thread(
                target=watch_child,
                args=(ev.session_thread_id, ev.agent_name),
                daemon=True,
            )
            watcher.start()
            watchers.append(watcher)
        elif ev.type == "agent.thread_message_sent":
            say("unit_planner", f"\n[task sent to {ev.to_agent_name}]")
        elif ev.type == "agent.thread_message_received":
            say("unit_planner", f"\n[{ev.from_agent_name} reported back]")
        elif ev.type == "agent.tool_use" and not ev.session_thread_id:
            # session_thread_id는 child thread에서 교차 게시된 호출을 표시합니다.
            # 해당 child의 watcher가 이미 보여 줍니다
            say("unit_planner", f"\n[{ev.name}]")
        elif ev.type == "session.status_terminated":
            break
        elif ev.type == "session.status_idle":
            if ev.stop_reason.type != "end_turn":
                say("unit_planner", f"\n[idle without end_turn: {ev.stop_reason.type}]")
            break

for watcher in watchers:
    watcher.join(timeout=60)
if any(watcher.is_alive() for watcher in watchers):
    print("\n[a child stream is still open after the join timeout]")

stream이 연결되지만 buffer에 쌓인 event만 전달한다면 `event_deltas` parameter가 거부되는 대신 제거된 것입니다. 이는 beta header가 기능을 제공하지 않는다는 징후입니다. thread stream URL의 404는 경로가 잘못되었거나 managed-agents beta header가 없음을 뜻합니다(thread endpoint는 beta로 제한됨). 경로는 `/threads/{thread_id}/stream`이며 `/threads/{thread_id}/events/stream`이 아닙니다.

전체 streaming contract(delta 형태, reconciliation 보장, 문제 해결 표)는 [event와 streaming](https://platform.claude.com/docs/en/managed-agents/events-and-streaming#event-deltas)에 있습니다.


## 6. preview와 기록 대조하기

preview의 delta를 이어 붙이면 buffer에 쌓인 event text의 prefix가 됩니다. 부하가 높으면 delta가 버려질 수 있으므로 반드시 전체 text는 아닙니다. 이 보장 때문에 reconciliation을 한 번의 교체로 수행할 수 있습니다. 아래 각 행은 buffer에 쌓인 `agent.message` 하나와 기록 도착 시 accumulator가 만든 preview의 비교입니다.


In [ ]:
for agent_name, preview_chars, final_chars, is_prefix in reconciliations:
    print(
        f"{agent_name:22s} preview {preview_chars:5d} chars | "
        f"final {final_chars:5d} chars | preview is prefix: {is_prefix}"
    )

## 7. 단원 계획 읽기

coordinator의 prompt는 전체 파일 `write` 호출로 계획을 작성하라고 지시하므로 event log에 전체 문서가 있으며 파일에 대한 마지막 `write`가 최종 version입니다. 자체 agent가 `edit`으로 파일을 수정한다면 write를 replay하는 대신 파일 자체를 읽으세요.


In [ ]:
unit_plan = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("unit_plan.md")
    ):
        unit_plan = ev.input["content"]  # 수정이 있을 수 있으므로 마지막 write를 보관합니다

# 전체 계획 대신 구조를 보여 줍니다. 전부 읽으려면 unit_plan을 출력하세요.
for line in unit_plan.splitlines():
    if line.startswith("#"):
        print(line)

## 8. version 왕복 호출 없이 prompt 변경 출시하기

review feedback이 모든 교육 기준 정합성 주장 옆에 기준 코드를 넣으라고 한다고 가정하세요. `agents.update`는 agent의 현재 `version`을 선택적인 concurrency key로 취급합니다. 생략하면 update는 조건 없이 적용되고 서버가 version을 올려 줍니다. provisioning script가 agent를 다시 쓰기 위해 읽을 필요가 없습니다.

동시 작성자가 있을 수 있으면 `version`을 제공하세요(불일치하면 409를 반환하므로 항상 알려진 상태에서 update합니다). commit된 agent 정의를 동기화하는 CI job처럼 하나의 흐름이 agent를 관리하면 생략하세요.


In [ ]:
updated = client.beta.agents.update(
    standards_researcher.id,
    system=RESEARCHER_SYSTEM
    + "\nCite the performance expectation code (for example MS-ESS2-4) next to every "
    "alignment claim.",
    betas=BETAS,
)
print(f"standards_researcher v{standards_researcher.version} -> v{updated.version}")

multiagent 설정의 주의점 하나입니다. coordinator roster는 coordinator 생성 시 researcher의 version을 고정했으므로 기존 coordinator는 계속 이전 version에 위임합니다. 최신 child version으로 roster를 다시 결정하려면 coordinator를 update하세요(`multiagent` 필드나 아무 필드).


## 9. 정리하기


In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")